## Comparison between the PPP and Overton peatland policy collections

### Overton Data Acquisition

A dataset with the names of 10,000 policies was exported from Overton on July 12th 2026 using the following criteria:
Search string: Peatland OR Peat OR Wetland
Search results filtered by:
- Source sector – Public sector
-	Source Organisation type – Government, Legislative Body
-	Years – 2000 to 2026
-	Source Country: Every EU country and the UK
-	Document type: Publication
Sorted by: Relevance

This search and subsequent filtering returned 34,872 policies, of which the top 10,000 most relevant were exported into "Overton data NEW.csv".



In [ ]:
!! pip install pandas thefuzz tqdm

['zsh:1: command not found: pip']

In [ ]:
import pandas as pd
from thefuzz import process as fuzzy_process
from tqdm import tqdm
import re
import os

cwd = os.getcwd()

# --- V9 FINAL CONFIGURATION ---
ODOO_NEW_PATH = f'{cwd}/Data/Odoo data NEW.csv'
OVERTON_NEW_PATH = f'{cwd}/Data/Overton data NEW.csv'
FINAL_OUTPUT_PATH_SUMMARY = f'{cwd}/Outputs/ppp_overton_level_summary.csv'
FINAL_OUTPUT_PATH_DETAILS = f'{cwd}/Outputs/ppp_overton_level_details.csv'

FUZZY_MATCH_THRESHOLD = 95
PUBLISHED_STATE_VALUE = 'Published'

# List of primary countries for consolidation.
# This ensures we correctly group subdivisions.
EU27_UK_COUNTRIES = [
    'Austria', 'Belgium', 'Bulgaria', 'Croatia', 'Cyprus', 'Czech Republic',
    'Denmark', 'Estonia', 'Finland', 'France', 'Germany', 'Greece', 'Hungary',
    'Ireland', 'Italy', 'Latvia', 'Lithuania', 'Luxembourg', 'Malta',
    'Netherlands', 'Poland', 'Portugal', 'Romania', 'Slovakia', 'Slovenia',
    'Spain', 'Sweden', 'United Kingdom'
]

def consolidate_overton_country(country_name: str) -> str:
    """
    Consolidates Overton's subdivided country names into a single parent country.
    Example: 'Germany: Bavaria' -> 'Germany'
    """
    if not isinstance(country_name, str):
        return 'Unknown'
    # Special handling for UK, as it's often used as a prefix.
    if country_name.startswith('UK'):
        return 'United Kingdom'
    for parent_country in EU27_UK_COUNTRIES:
        if country_name.startswith(parent_country):
            return parent_country
    # If it's not a subdivision of our target countries, return it as is (e.g., 'EU').
    return country_name

def attempt_reciprocal_match(
    ppp_search_title: str,
    candidate_overton_titles: list,
    candidate_ppp_titles: list,
    threshold: int
) -> dict:
    """
    Performs a high-confidence reciprocal match.
    """
    if not isinstance(ppp_search_title, str) or not ppp_search_title.strip() or not candidate_overton_titles:
        return None

    forward_match = fuzzy_process.extractOne(ppp_search_title, candidate_overton_titles)

    if forward_match and forward_match[1] >= threshold:
        best_overton_title, score = forward_match
        if candidate_ppp_titles:
            reverse_match = fuzzy_process.extractOne(best_overton_title, candidate_ppp_titles)
            if reverse_match and reverse_match[0] == ppp_search_title:
                return {'overton_title': best_overton_title, 'similarity_score': score}
    return None

Main analysis script to determine the coverage of different policy levels in Overton.

In [ ]:

print("--- Starting Analysis (V9 - Policy Level Coverage) ---")

# 1. Load and Prepare Odoo (PPP) Data
try:
    ppp_df = pd.read_csv(ODOO_NEW_PATH, usecols=['Country', 'Name', 'Name (Native Language)', 'State', 'Level'], engine='python')
    ppp_df.rename(columns={'Name': 'title_en', 'Name (Native Language)': 'title_native'}, inplace=True)
    ppp_df_published = ppp_df[ppp_df['State'] == PUBLISHED_STATE_VALUE].copy()
    # Fill NaN countries with a placeholder for grouping.
    ppp_df_published['Country'].fillna('No Country', inplace=True)
    print(f"Loaded and filtered {len(ppp_df_published)} 'Published' policies from Odoo.")
except Exception as e:
    print(f"Fatal Error loading Odoo data: {e}")

--- Starting Analysis (V9 - Policy Level Coverage) ---
Loaded and filtered 571 'Published' policies from Odoo.


/var/folders/b_/qtqdyvg96nn9w7r05syyg8pm0000gn/T/ipykernel_11378/2736396949.py:9: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  ppp_df_published['Country'].fillna('No Country', inplace=True)


In [ ]:
# 2. Load and Prepare Overton Data
try:
    overton_df = pd.read_csv(OVERTON_NEW_PATH, usecols=['Title', 'Source country'], engine='python', on_bad_lines='warn')
    overton_df.rename(columns={'Title': 'overton_title', 'Source country': 'overton_country'}, inplace=True)
    overton_df.dropna(subset=['overton_title', 'overton_country'], inplace=True)
    # Apply the consolidation logic to the Overton country data.
    overton_df['consolidated_country'] = overton_df['overton_country'].apply(consolidate_overton_country)
    overton_titles_by_country = overton_df.groupby('consolidated_country')['overton_title'].apply(list).to_dict()
    print(f"Processed {len(overton_df)} policies from Overton and consolidated country names.")
except Exception as e:
    print(f"Fatal Error loading Overton data: {e}")

/var/folders/b_/qtqdyvg96nn9w7r05syyg8pm0000gn/T/ipykernel_11378/503435354.py:3: ParserWarning: Skipping line 14700: ',' expected after '"'

  overton_df = pd.read_csv(OVERTON_NEW_PATH, usecols=['Title', 'Source country'], engine='python', on_bad_lines='warn')
/var/folders/b_/qtqdyvg96nn9w7r05syyg8pm0000gn/T/ipykernel_11378/503435354.py:3: ParserWarning: Skipping line 21421: ',' expected after '"'

  overton_df = pd.read_csv(OVERTON_NEW_PATH, usecols=['Title', 'Source country'], engine='python', on_bad_lines='warn')
/var/folders/b_/qtqdyvg96nn9w7r05syyg8pm0000gn/T/ipykernel_11378/503435354.py:3: ParserWarning: Skipping line 22804: ',' expected after '"'

  overton_df = pd.read_csv(OVERTON_NEW_PATH, usecols=['Title', 'Source country'], engine='python', on_bad_lines='warn')
/var/folders/b_/qtqdyvg96nn9w7r05syyg8pm0000gn/T/ipykernel_11378/503435354.py:3: ParserWarning: Skipping line 23308: ',' expected after '"'

  overton_df = pd.read_csv(OVERTON_NEW_PATH, usecols=['Title', 'Source count

Processed 39244 policies from Overton and consolidated country names.


In [15]:
# 3. Perform Matching
# [takes about 2 minutes]
results = []
print("\nBeginning cross-referencing...")
for _, ppp_row in tqdm(ppp_df_published.iterrows(), total=len(ppp_df_published), desc="Matching Policies"):
    ppp_country = ppp_row['Country']
    match_found = False

    # Define the search spaces for this PPP policy
    search_countries = []
    if ppp_country in EU27_UK_COUNTRIES:
        search_countries.append(ppp_country)
    search_countries.append('EU') # Always search the 'EU' bucket as requested

    # Get all possible PPP titles for reciprocal check
    candidate_ppp_titles = [t for t in [ppp_row['title_en'], ppp_row['title_native']] if pd.notna(t)]

    for search_country in search_countries:
        candidate_overton_titles = overton_titles_by_country.get(search_country, [])
        if not candidate_overton_titles:
            continue

        for ppp_title in candidate_ppp_titles:
            match_info = attempt_reciprocal_match(ppp_title, candidate_overton_titles, candidate_ppp_titles, FUZZY_MATCH_THRESHOLD)
            if match_info:
                match_found = True
                break
        if match_found:
            break
    
    results.append({
        'ppp_country': ppp_country,
        'ppp_level': ppp_row['Level'],
        'match_status': 'Match' if match_found else 'No Match'
    })

results_df = pd.DataFrame(results)


Beginning cross-referencing...


Matching Policies:  28%|██▊       | 159/571 [00:11<00:31, 13.28it/s]


KeyboardInterrupt: 

In [ ]:
results_df

,ppp_country,ppp_level,match_status
0,No Country,Global,No Match
1,No Country,Global,No Match
2,No Country,Global,No Match
3,No Country,Global,No Match
4,No Country,Global,No Match
...,...,...,...
566,France,National,No Match
567,France,National,No Match
568,France,National,No Match
569,Malta,National,No Match


Generate and save output tables

In [ ]:
# Table 1: Summary of matches by policy level (unchanged)
level_summary = results_df.groupby('ppp_level')['match_status'].value_counts().unstack(fill_value=0)
level_summary['Total'] = level_summary.sum(axis=1)
level_summary['Match_Percentage'] = (level_summary['Match'] / level_summary['Total'] * 100).round(2)

print("\n--- Table 1: Overton Match Rate by Policy Level ---")
print(level_summary.to_string())
level_summary.to_csv(FINAL_OUTPUT_PATH_SUMMARY)
print(f"Summary table saved to: {FINAL_OUTPUT_PATH_SUMMARY}")


--- Table 1: Overton Match Rate by Policy Level ---
match_status  Match  No Match  Total  Match_Percentage
ppp_level                                             
European          0        21     21              0.00
Global            0        10     10              0.00
Local             1        32     33              3.03
National         53       344    397             13.35
Regional         12        98    110             10.91
Summary table saved to: c:\Users\Ales\Documents\GitHub\PPP_v_Overton_Data_Analysis/Outputs/ppp_overton_level_summary.csv


In [ ]:
# Table 2: Updated to be a summary by country only
country_summary = results_df.groupby('ppp_country')['match_status'].value_counts().unstack(fill_value=0)
# Ensure 'Match' and 'No Match' columns exist to prevent errors if one is missing
if 'Match' not in country_summary:
    country_summary['Match'] = 0
if 'No Match' not in country_summary:
    country_summary['No Match'] = 0

country_summary['Total'] = country_summary['Match'] + country_summary['No Match']
country_summary['Match_Percentage'] = (country_summary['Match'] / country_summary['Total'] * 100).round(2)

print("\n--- Table 2: Detailed Breakdown by Country ---")
# We can drop the 'No Country' category from the final display table for clarity
print(country_summary.drop('No Country', errors='ignore').to_string())
country_summary.to_csv(FINAL_OUTPUT_PATH_DETAILS)
print(f"Detailed breakdown saved to: {FINAL_OUTPUT_PATH_DETAILS}")


--- Table 2: Detailed Breakdown by Country ---
match_status    Match  No Match  Total  Match_Percentage
ppp_country                                             
Austria             2         7      9             22.22
Belgium             0        25     25              0.00
Bulgaria            4         9     13             30.77
Croatia             1         8      9             11.11
Cyprus              0        16     16              0.00
Czech Republic      1        13     14              7.14
Denmark             1        10     11              9.09
Estonia             4        20     24             16.67
Finland             1        23     24              4.17
France              0        21     21              0.00
Germany             8        58     66             12.12
Greece              0        17     17              0.00
Hungary             0         9      9              0.00
Ireland            12        23     35             34.29
Italy               1        30     31  